# Семинар 04. Из функции в проект

Создаём устанавливаемый пакет с CLI, проверками и воспроизводимой Git-историей.

## Цели

- создать репозиторий и виртуальное окружение;
- различать модуль, пакет и импорт;
- собрать проект в `src`-layout;
- описать метаданные и инструменты в `pyproject.toml`;
- установить пакет в editable-режиме;
- отделить ядро, I/O и CLI;
- добавить console entry point, Ruff и smoke-тест;
- выполнить работу в ветках и разрешить merge conflict;
- записать воспроизводимые команды в README.

## Перед началом

Основную работу выполняйте в терминале и редакторе в отдельном учебном репозитории. Команды ниже записаны для macOS/Linux; в Windows меняется прежде всего команда активации окружения. Не копируйте `.venv` и не добавляйте его в Git.

В качестве учебного имени используется `search-tool` для дистрибутива и `search_tool` для импортируемого пакета. Группа может подставить своё корректное имя.

## Результат семинара

К концу занятия репозиторий должен выглядеть так:

```text
search-tool/
├── .gitignore
├── pyproject.toml
├── README.md
├── src/
│   └── search_tool/
│       ├── __init__.py
│       ├── algorithms.py
│       ├── io.py
│       └── cli.py
└── tests/
    └── test_smoke.py
```

Предметная функция не знает о файлах и CLI, пакет устанавливается, команда `search-tool` запускается, а проверки воспроизводятся из README.

## Задание 1. Имя и репозиторий

Имя дистрибутива может содержать дефис, но имя Python-пакета должно быть допустимым идентификатором. Создайте пустой каталог и репозиторий:

```bash
mkdir search-tool
cd search-tool
git init
git branch -M main
```

Сразу создайте `.gitignore` и исключите как минимум `.venv/`, `__pycache__/`, `.pytest_cache/`, `.ruff_cache/`, `.DS_Store` и настройки редактора, если они не являются общими для команды.

In [ ]:
distribution_name = "search-tool"
package_name = "search_tool"

# TODO: подставьте имена своего проекта.
assert package_name.isidentifier()
assert "-" not in package_name
print(distribution_name, package_name)

## Задание 2. Чистое окружение

Виртуальное окружение изолирует интерпретатор и установленные зависимости проекта:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python --version
python -m pip --version
```

В Windows PowerShell активация обычно выполняется командой `.venv\Scripts\Activate.ps1`. После активации `python` и `pip` должны указывать внутрь `.venv`.

Окружение является воспроизводимым артефактом, а не исходным кодом: в Git хранят описание зависимостей, но не каталог `.venv`.

## Модуль, пакет и импорт

- файл `algorithms.py` — модуль;
- каталог `search_tool/` с `__init__.py` — пакет;
- `from search_tool.algorithms import linear_search` — импорт имени из модуля пакета.

Импорт выполняет код верхнего уровня модуля. Поэтому определения функций и констант размещают наверху, а интерактивный запуск защищают условием `if __name__ == "__main__":`. Устанавливаемый CLI лучше оформлять отдельным entry point.

## Зачем нужен `src`-layout

При плоской структуре запуск из корня может случайно импортировать исходный каталог, даже если пакет не устанавливается. `src`-layout заставляет проверить именно установленный пакет и уменьшает число таких ложноположительных импортов.

Создайте каталоги и пустые файлы удобным для вашей системы способом:

```text
src/search_tool/__init__.py
src/search_tool/algorithms.py
src/search_tool/io.py
src/search_tool/cli.py
tests/test_smoke.py
```

После этого зафиксируйте каркас отдельным коммитом.

In [ ]:
from pathlib import Path

project_root = Path("search-tool")  # TODO: укажите свой путь.
package_name = "search_tool"
required = [
    project_root / "pyproject.toml",
    project_root / "README.md",
    project_root / "src" / package_name / "__init__.py",
    project_root / "src" / package_name / "algorithms.py",
    project_root / "src" / package_name / "io.py",
    project_root / "src" / package_name / "cli.py",
    project_root / "tests" / "test_smoke.py",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, missing

## Задание 3. Перенесите предметное ядро

В `algorithms.py` перенесите одну функцию из предыдущего занятия. Для примера используем контракт:

```python
def linear_search(values: list[int], target: int) -> int:
    ...
```

Модуль ядра не импортирует `argparse`, не читает файлы и ничего не печатает. Он принимает значения и возвращает значение. Это позволяет использовать одну реализацию из CLI, ноутбука, теста или будущего HTTP API.

In [ ]:
def linear_search(values: list[int], target: int) -> int:
    # TODO: перенесите и проверьте свою реализацию.
    return -1


assert linear_search([], 5) == -1
assert linear_search([1, 3, 5], 1) == 0
assert linear_search([1, 3, 5], 5) == 2
assert linear_search([1, 3, 5], 4) == -1

## Граница файлового ввода

`io.py` переводит внешний файл в значения Python. Он может содержать функцию:

```python
from pathlib import Path


def read_integers(path: Path) -> list[int]:
    values = []
    for line_number, line in enumerate(
        path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        stripped = line.strip()
        if not stripped:
            continue
        try:
            values.append(int(stripped))
        except ValueError as error:
            raise ValueError(
                f"line {line_number}: expected integer"
            ) from error
    return values
```

Функция не решает, как показать ошибку пользователю. Она добавляет контекст и передаёт исключение внешнему слою.

## `pyproject.toml`

Минимальная конфигурация проекта:

```toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "search-tool"
version = "0.1.0"
description = "Search an integer in a text file"
readme = "README.md"
requires-python = ">=3.11"
dependencies = []

[project.optional-dependencies]
dev = ["pytest>=8", "ruff>=0.8"]

[project.scripts]
search-tool = "search_tool.cli:main"

[tool.pytest.ini_options]
testpaths = ["tests"]

[tool.ruff]
line-length = 88

[tool.ruff.lint]
select = ["E", "F", "I"]
```

`[project]` описывает устанавливаемый дистрибутив, `[project.scripts]` связывает команду с функцией, а секции `[tool.*]` хранят настройки инструментов.

## Editable-установка

Установите пакет вместе с инструментами разработки:

```bash
python -m pip install -e ".[dev]"
python -c "import search_tool; print(search_tool.__file__)"
```

Флаг `-e` связывает установленный пакет с исходным каталогом: после изменения `.py`-файлов переустановка обычно не нужна. Метаданные и entry points при изменении `pyproject.toml` могут потребовать повторной установки.

Импорт через ручную подмену `PYTHONPATH` не считается воспроизводимой установкой.

## CLI как тонкий адаптер

`cli.py` отвечает за аргументы, сообщения и код завершения:

```python
import argparse
import sys
from pathlib import Path

from search_tool.algorithms import linear_search
from search_tool.io import read_integers


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser()
    parser.add_argument("path", type=Path)
    parser.add_argument("target", type=int)
    return parser


def main() -> int:
    args = build_parser().parse_args()
    try:
        values = read_integers(args.path)
    except (FileNotFoundError, ValueError) as error:
        print(f"error: {error}", file=sys.stderr)
        return 2

    print(linear_search(values, args.target))
    return 0
```

CLI координирует готовые части, но не реализует поиск и разбор формата заново.

## Задание 4. Запустите entry point

После установки проверьте интерфейс:

```bash
search-tool --help
search-tool data/numbers.txt 42
echo $?
```

Создайте небольшой входной файл и проверьте три сценария: цель найдена, цели нет, файл отсутствует. Для успешных сценариев ожидается код `0`, для ошибки пользовательского ввода — ненулевой код.

Если команда не найдена, сначала проверьте активированное окружение и повторите editable-установку.

In [ ]:
scenarios = [
    # arguments, expected_stdout, expected_exit_code
    (["data/numbers.txt", "42"], "5", 0),
    (["data/numbers.txt", "100"], "-1", 0),
    (["missing.txt", "42"], "", 2),
]

# TODO: выполните эти сценарии вручную или через subprocess
# и сравните stdout, stderr и код завершения.
assert len(scenarios) == 3

## Smoke-тест

Smoke-тест быстро отвечает на вопрос «основной сценарий вообще запускается?». Он не заменяет подробные тесты занятия 10. Создайте `tests/test_smoke.py`:

```python
from search_tool.algorithms import linear_search


def test_package_core_smoke() -> None:
    assert linear_search([10, 20, 30], 20) == 1
```

Запустите `pytest`. Тест должен импортировать установленный пакет, а не файл через относительный путь к `src`.

## Ruff и единая последовательность проверки

Выполните из корня проекта:

```bash
ruff check .
ruff format --check .
pytest
search-tool --help
```

`ruff check` ищет выбранные классы проблем, `ruff format --check` проверяет формат без изменения файлов, `pytest` выполняет smoke-тест, последняя команда проверяет установленный CLI.

Если нужно применить форматирование, отдельно выполните `ruff format .`, просмотрите diff и повторите проверки.

In [ ]:
checks = [
    "ruff check .",
    "ruff format --check .",
    "pytest",
    "search-tool --help",
]

# TODO: после запуска отметьте код завершения каждой команды.
for command in checks:
    print("[ ]", command)

## Работа в ветках

Перед независимой доработкой создают ветку от актуальной основной ветки:

```bash
git status
git switch main
git switch -c feature/core
# изменить algorithms.py и тест
git add src/search_tool/algorithms.py tests/test_smoke.py
git commit -m "feat: add search core"
```

Для CLI создайте другую ветку от `main`, а не поверх незавершённой `feature/core`. Перед переключением рабочее дерево должно быть осознанно сохранено коммитом или оставаться чистым.

## Учебный merge conflict

Создайте две ветки, которые меняют одну строку описания в README по-разному. Затем слейте обе в `main`:

```bash
git switch main
git merge feature/description-a
git merge feature/description-b
```

В конфликтующем файле Git оставит маркеры:

```text
<<<<<<< HEAD
текст из текущей ветки
=======
текст из присоединяемой ветки
>>>>>>> feature/description-b
```

Прочитайте оба варианта, соберите правильный итоговый текст, удалите маркеры, затем выполните:

```bash
git add README.md
git commit
```

Разрешить конфликт — значит принять содержательное решение, а не механически выбрать «наш» или «их» файл.

In [ ]:
conflict_markers = ["<<<<<<<", "=======", ">>>>>>>"]
readme_text = ""  # TODO: вставьте или прочитайте итоговый README.

remaining = [marker for marker in conflict_markers if marker in readme_text]
assert not remaining, remaining

## README как инструкция воспроизведения

Новый участник команды должен суметь запустить проект без устных дополнений. Запишите:

1. назначение и минимальный сценарий;
2. требуемую версию Python;
3. создание и активацию окружения;
4. установку `python -m pip install -e ".[dev]"`;
5. пример CLI с ожидаемым результатом;
6. команды `ruff` и `pytest`;
7. известные ограничения.

Проверьте README в новом окружении или попросите другую группу выполнить инструкцию буквально.

## Минимальная воспроизводимая последовательность

Из чистого клона должно быть достаточно выполнить:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e ".[dev]"
ruff check .
ruff format --check .
pytest
search-tool --help
```

Если последовательность требует ручного копирования файлов, изменения `PYTHONPATH` или установки неописанного пакета, проект пока невоспроизводим.

## Контрольная точка проекта

До следующего занятия группа фиксирует в репозитории:

- пользователя и проверяемую проблему;
- вход, выход и границы первой версии;
- минимальное предметное ядро;
- `pyproject.toml`, `src`-layout и инструкцию запуска;
- распределение ответственности;
- ближайшую небольшую техническую задачу для каждого участника.

Не добавляйте базу данных, веб-фреймворк или ИИ только ради списка технологий. Они появляются, когда этого требует сценарий.

## Полезные материалы

- [Packaging Python Projects — Python Packaging User Guide](https://packaging.python.org/en/latest/tutorials/packaging-projects/)
- [src layout vs flat layout — Python Packaging User Guide](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/)
- [Configuring Ruff](https://docs.astral.sh/ruff/configuration/)
- [pytest documentation](https://docs.pytest.org/)
- [Pro Git: Git Branching](https://git-scm.com/book/en/v2/Git-Branching-Branches-in-a-Nutshell)

## Самопроверка

1. Чем модуль отличается от пакета?
2. Что хранится в Git вместо каталога `.venv`?
3. Какую проблему уменьшает `src`-layout?
4. Что описывают `[project]` и `[project.scripts]`?
5. Почему CLI не должен реализовывать предметный алгоритм?
6. Что проверяет smoke-тест?
7. Чем `ruff check` отличается от `ruff format --check`?
8. Когда нужно повторить editable-установку?
9. Что означает содержательно разрешить merge conflict?
10. Какие команды должны работать из чистого клона?

## Итоги

- Виртуальное окружение изолирует зависимости, а `pyproject.toml` описывает проект.
- `src`-layout отделяет исходный код от корня репозитория.
- Предметное ядро независимо от файлов и CLI.
- Console entry point создаёт устанавливаемую команду.
- Ruff и smoke-тест образуют минимальный контур проверки.
- Ветки изолируют доработки, а конфликт требует осмысленного объединения.
- README превращает набор файлов в воспроизводимый проект.

Закрепите результат в [самостоятельной работе](tasks.md) или непосредственно в репозитории группового проекта.